## Retrieving the dataset from the UCI Machine Learning Repository
pip install ucimlrepo

In [5]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
myocardial_infarction_complications = fetch_ucirepo(id=579) 
  
# data (as pandas dataframes) 
features_X = myocardial_infarction_complications.data.features 
targets_Y = myocardial_infarction_complications.data.targets 


## Generate Human Readable Metadata and Variable Information

In [13]:
import json
with open("metadata_pretty.json", "w") as f:
    json.dump(
        myocardial_infarction_complications.metadata,
        f,
        indent=4
    )

In [ ]:
variables_info_df = myocardial_infarction_complications.variables

variables_info_df.to_csv("variables_info.csv", index=False)

In [10]:
import pandas as pd

html_table = (
    variables_info_df.style
    .hide(axis="index")
    .set_caption("Data Dictionary")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "18px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "8px 0")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#f2f2f2"),
                ("padding", "8px"),
                ("text-align", "left"),
                ("border", "1px solid #ddd")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px"),
                ("border", "1px solid #ddd"),
                ("vertical-align", "top")
            ]
        }
    ])
    .to_html()
)

with open("data_dictionary.html", "w", encoding="utf-8") as f:
    f.write(html_table)

print("Saved data_dictionary.html")

Saved data_dictionary.html


## Saving the Raw DataFrame as a Parquet file for future use

In [29]:
import pandas as pd
# conda install pyarrow

full_data = pd.concat([features_X, targets_Y], axis=1)

In [30]:
full_data.to_parquet("Myocardial_Infarction_Raw.parquet", engine="pyarrow")

## Likely focusing on The target variable "ZSN" 

* ZSN is a binary outcome variable indicating whether a patient developed chronic heart failure following myocardial infarction.
* Among the available outcomes, it exhibits a relatively balanced class distribution (i.e., a sufficient number of positive cases), making it well-suited for supervised machine learning without extreme class imbalance issues.
* It is also clinically meaningful, as heart failure is a common and serious complication after myocardial infarction, with clear implications for patient prognosis and management.

In [31]:
import numpy as np
np.random.seed(321)

full_data_split = full_data.copy() # defragmenting the dataframe to avoid SettingWithCopyWarning
full_data_split["train_dummy"] = (np.random.rand(len(full_data_split)) < 0.75).astype(int)

In [32]:
cols = [c for c in full_data_split.columns if c not in ["train_dummy", "ZSN"]]

full_data_split = full_data_split[["train_dummy", "ZSN"] + cols]

In [33]:
full_data_split.to_parquet("Myocardial_Infarction_Split.parquet", engine="pyarrow")